# Hyperparameter Optimization

This notebook demonstrates hyperparameter tuning using scikit-learn's built-in search methods on real datasets.

## Topics Covered
1. GridSearchCV - exhaustive search
2. RandomizedSearchCV - random sampling
3. Visualization of parameter search spaces and results
4. Cross-validation analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine, load_iris
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV, cross_val_score, StratifiedKFold
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.stats import randint, uniform
import time
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

## Load Dataset

In [ ]:
# Load the wine dataset
wine = load_wine()
X, y = wine.data, wine.target

print(f"Dataset: Wine Recognition")
print(f"Samples: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Classes: {len(np.unique(y))} ({np.bincount(y)})")
print(f"Feature names: {wine.feature_names}")

---
## 1. GridSearchCV - Exhaustive Search

GridSearchCV tries **every combination** of the specified hyperparameters. Guaranteed to find the best combination in the grid, but can be slow with large search spaces.

In [ ]:
# Define the parameter grid for Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

total_combinations = 1
for values in param_grid.values():
    total_combinations *= len(values)
print(f"Total parameter combinations: {total_combinations}")
print(f"With 5-fold CV: {total_combinations * 5} model fits\n")

# Run GridSearchCV
rf = RandomForestClassifier(random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

start_time = time.time()
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    return_train_score=True
)
grid_search.fit(X, y)
grid_time = time.time() - start_time

print(f"GridSearchCV completed in {grid_time:.2f} seconds")
print(f"Best accuracy: {grid_search.best_score_:.4f}")
print(f"Best parameters: {grid_search.best_params_}")

---
## 2. RandomizedSearchCV - Random Sampling

Instead of trying all combinations, RandomizedSearchCV **samples** a fixed number of parameter settings. More efficient for large search spaces.

In [ ]:
# Define distributions for random sampling
param_distributions = {
    'n_estimators': randint(50, 300),
    'max_depth': [3, 5, 10, 15, 20, None],
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': uniform(0.1, 0.9)
}

n_iter = 50  # Number of random combinations to try
print(f"Random search: {n_iter} random combinations (from a vast space)\n")

start_time = time.time()
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_distributions,
    n_iter=n_iter,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    return_train_score=True
)
random_search.fit(X, y)
random_time = time.time() - start_time

print(f"RandomizedSearchCV completed in {random_time:.2f} seconds")
print(f"Best accuracy: {random_search.best_score_:.4f}")
print(f"Best parameters: {random_search.best_params_}")

---
## 3. Comparison: Grid vs Random Search

In [ ]:
comparison = pd.DataFrame({
    'Method': ['GridSearchCV', 'RandomizedSearchCV'],
    'Best Accuracy': [grid_search.best_score_, random_search.best_score_],
    'Time (seconds)': [grid_time, random_time],
    'Combinations Tried': [total_combinations, n_iter]
})

print(comparison.to_string(index=False))

---
## 4. Visualize Parameter Search Results

In [ ]:
# Extract results from RandomizedSearchCV for richer visualization
results = pd.DataFrame(random_search.cv_results_)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# n_estimators vs accuracy
axes[0, 0].scatter(results['param_n_estimators'], results['mean_test_score'],
                    c=results['mean_test_score'], cmap='viridis', alpha=0.7, edgecolors='black')
axes[0, 0].set_xlabel('n_estimators')
axes[0, 0].set_ylabel('Mean CV Accuracy')
axes[0, 0].set_title('n_estimators vs Accuracy')

# max_depth vs accuracy
depth_results = results.copy()
depth_results['param_max_depth'] = depth_results['param_max_depth'].apply(
    lambda x: 25 if x is None else x
)
axes[0, 1].scatter(depth_results['param_max_depth'], results['mean_test_score'],
                    c=results['mean_test_score'], cmap='viridis', alpha=0.7, edgecolors='black')
axes[0, 1].set_xlabel('max_depth (25 = None)')
axes[0, 1].set_ylabel('Mean CV Accuracy')
axes[0, 1].set_title('max_depth vs Accuracy')

# min_samples_split vs accuracy
axes[1, 0].scatter(results['param_min_samples_split'], results['mean_test_score'],
                    c=results['mean_test_score'], cmap='viridis', alpha=0.7, edgecolors='black')
axes[1, 0].set_xlabel('min_samples_split')
axes[1, 0].set_ylabel('Mean CV Accuracy')
axes[1, 0].set_title('min_samples_split vs Accuracy')

# Distribution of CV scores
axes[1, 1].hist(results['mean_test_score'], bins=15, color='steelblue', edgecolor='black', alpha=0.7)
axes[1, 1].axvline(random_search.best_score_, color='red', linestyle='--', linewidth=2, label=f'Best: {random_search.best_score_:.4f}')
axes[1, 1].set_xlabel('Mean CV Accuracy')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Distribution of CV Scores')
axes[1, 1].legend()

plt.suptitle('Hyperparameter Search Results (RandomizedSearchCV)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Cross-Validation Results Analysis

In [ ]:
# Detailed cross-validation for the best model
best_model = random_search.best_estimator_
cv_scores = cross_val_score(best_model, X, y, cv=cv, scoring='accuracy')

print(f"Best model cross-validation results:")
print(f"  Scores per fold: {cv_scores}")
print(f"  Mean accuracy:   {cv_scores.mean():.4f}")
print(f"  Std deviation:   {cv_scores.std():.4f}")
print(f"  95% CI:          [{cv_scores.mean() - 2*cv_scores.std():.4f}, {cv_scores.mean() + 2*cv_scores.std():.4f}]")

# Compare train vs test scores (check for overfitting)
top_n = 10
top_results = results.nlargest(top_n, 'mean_test_score')[[
    'mean_test_score', 'std_test_score', 'mean_train_score', 'std_train_score', 'mean_fit_time'
]].reset_index(drop=True)
top_results.index = range(1, top_n + 1)
top_results.index.name = 'Rank'

print(f"\nTop {top_n} configurations:")
print(top_results.round(4).to_string())

In [ ]:
# Train vs Test score comparison for overfitting detection
fig, ax = plt.subplots(figsize=(10, 6))

sorted_results = results.sort_values('mean_test_score', ascending=True).tail(30)

x_pos = range(len(sorted_results))
ax.barh(x_pos, sorted_results['mean_train_score'], height=0.4, 
        label='Train Score', color='lightcoral', alpha=0.8)
ax.barh([x + 0.4 for x in x_pos], sorted_results['mean_test_score'], height=0.4,
        label='Test Score', color='steelblue', alpha=0.8)

ax.set_xlabel('Accuracy', fontsize=12)
ax.set_ylabel('Configuration Rank', fontsize=12)
ax.set_title('Train vs Test Accuracy (Top 30 Configurations)', fontsize=14)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("If train >> test: overfitting. If train ~= test: good generalization.")

## Key Takeaways

1. **GridSearchCV** is exhaustive but expensive - use for small parameter spaces.
2. **RandomizedSearchCV** is more efficient and often finds comparable results with fewer evaluations.
3. Always use **cross-validation** to get reliable performance estimates.
4. Monitor **train vs test scores** to detect overfitting.
5. Visualize results to understand which hyperparameters matter most.